In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
import os
os.chdir("./..")

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

pd.set_option("display.max_columns", None)

# Loading data

In [ ]:
alarms = pd.read_csv("data/alarms/alarms_data_preprocessed_v1.csv")
weather = pd.read_csv("data/weather/weather_data_preprocessed_v2.csv")
isw = pd.read_csv("data/isw/isw_data_preprocessed_v3.csv")
telegram = pd.read_csv("data/telegram/telegram_hourly_features_v3.csv")

In [ ]:
alarms.head(3)

In [ ]:
weather.head(3)

In [ ]:
isw.head(3)

In [ ]:
telegram.head(3)

# Merge problem

# Alarms data

In [ ]:
alarms.head()

In [ ]:
alarms.info()

Convert `start`, `end` columns to datetime format

In [ ]:
alarms.drop(["start_hour", "day_of_week", "date", "month_year", "duration_min"], axis=1, inplace=True)

In [ ]:
alarms["start"] = pd.to_datetime(alarms["start"])
alarms["end"] = pd.to_datetime(alarms["end"])

In [ ]:
alarms

In [ ]:
alarms.isna().sum()

In [ ]:
timeline = pd.date_range(
    alarms.start.min().floor("h"),
    alarms.end.max().ceil("h"),
    freq="h"
)

region_id = alarms.region_id.unique()

In [ ]:
spine = pd.MultiIndex.from_product([region_id, timeline], names=["region_id", "time"]) \
            .to_frame(index=False) \
            .sort_values(["region_id", "time"]) \
            .reset_index(drop=True)

In [ ]:
spine.head()

In [ ]:
# create a helper column with the list of hours per alarm
alarms["hours"] = alarms.apply(
    lambda row: pd.date_range(row["start"].floor("h"), row["end"].floor("h"), freq="h"),
    axis=1
)

# explode - each hour becomes its own row, region_id stays attached
alarm_expanded = alarms[["region_id", "hours"]].explode("hours")
alarm_expanded = alarm_expanded.rename(columns={"hours": "time"})
alarm_expanded["alarm"] = 1

# merge onto spine
merged_df = spine.merge(alarm_expanded, on=["region_id", "time"], how="left")
merged_df["alarm"] = merged_df["alarm"].fillna(0).astype(int)

In [ ]:
merged_df.loc[merged_df.region_id == 12].head()

In [ ]:
weather.head()

In [ ]:
weather.isna().sum()

In [ ]:
regions = pd.read_csv("data/alarms/regions.csv")

regions

In [ ]:
regions.center_city_en.unique()

In [ ]:
weather_df = weather.copy()

In [ ]:
weather_df.city.unique()

In [ ]:
regions.loc[~regions.center_city_en.isin(set(weather_df.city.unique()))]

There is no Crimea and Luhansk in `weather` dataset. But knowing that in this cities permanent alarm status, it is not necessary to know weather there. Model should always predict 1 for alarm status there.

In [ ]:
region_id = pd.DataFrame({"city": regions.center_city_en, "region_id": regions.region_id})
weather_df = weather_df.merge(region_id, on=["city"], how="left")

In [ ]:
weather_df["time"] = pd.to_datetime(weather_df.pop("real_hour_datetime"))

In [ ]:
weather_df

In [ ]:
merged_df = merged_df.merge(weather_df, on=["region_id", "time"], how="left")

In [ ]:
merged_df.info()

In [ ]:
tg_df = telegram.copy()
tg_df.rename({"datetime": "time"}, axis=1, inplace=True)
tg_df.time = pd.to_datetime(tg_df.time, utc=True) \
                .dt.tz_convert("Europe/Kyiv") \
                .dt.floor("h", ambiguous="infer")

In [ ]:
merged_df.time = merged_df.time.dt.tz_localize("Europe/Kyiv", ambiguous="NaT", nonexistent="NaT")

In [ ]:
merged_df = merged_df.merge(tg_df, on=["time"], how="left")

In [ ]:
merged_df["date"] = merged_df["time"].dt.date

In [ ]:
isw.info()

In [ ]:
isw.date = pd.to_datetime(isw.date)

In [ ]:
isw.isna().sum()

In [ ]:
isw = isw.loc[~isw.date.isna()]

In [ ]:
isw.isna().sum().sum()

In [ ]:
merged_df.date = pd.to_datetime(merged_df.date)

In [ ]:
merged_df = merged_df.merge(isw, on="date", how="left")

In [ ]:
merged_df.info()

In [ ]:
merged_df.shape

In [ ]:
df = merged_df.copy()

In [ ]:
df = df.loc[~df.time.isna()]

In [ ]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

In [ ]:
df = df.drop(["date", "city"], axis=1)

In [ ]:
df = df.loc[~df.hour_temp.isna()]

df.text_length = df.text_length.fillna(0)
df.hour_precip = df.hour_precip.fillna(0)
df.hour_preciptype = df.hour_preciptype.fillna("None")

df.hour_visibility = df.hour_visibility.ffill()
df.hour_uvindex = df.hour_uvindex.ffill()

In [ ]:
df.region_id.nunique()

In [ ]:
with pd.option_context("display.max_rows", None):
    print(df.isna().sum())

In [ ]:
df.region_id.nunique()

In [ ]:
df.loc[(df.cluster.isna() & df.text_length != 0)]

In [ ]:
df.region_id.nunique()

In [ ]:
df = df.fillna(-1)

In [ ]:
df.isna().sum().sum()

In [ ]:
df.region_id.nunique()

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
encoder = LabelEncoder()

cat_cols = list(df.select_dtypes(include=["str"]).columns)

for col_name in cat_cols:
    df[col_name] = encoder.fit_transform(df[[col_name]])

In [ ]:
df.time.isna().sum()

In [ ]:
df.head(3)

In [ ]:
df.shape

In [ ]:
df.region_id.nunique()

In [ ]:
scaler = StandardScaler()

num_cols = list(df.select_dtypes(include=["int", "float"]).columns)

df_scaled = pd.DataFrame(scaler.fit_transform(df[num_cols]), columns=num_cols)

In [ ]:
df_scaled.shape

In [ ]:
df = df_scaled.merge(df["time"], how="left", left_index=True, right_index=True)

In [ ]:
(df.time=="NaT").sum()

In [ ]:
df.shape

In [ ]:
df.sort_values(by=["region_id", "time"])

In [ ]:
df.time.isna().sum()

In [ ]:
save_path = Path("data/merged_v1.csv")

if not save_path.exists():
    df.to_csv(save_path, index=False)

In [ ]:
df = pd.read_csv(save_path)

In [ ]:
regions = df.region_id.unique()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit


tss = TimeSeriesSplit(n_splits=5, gap=24*7)

unique_hours = df["time"].sort_values().unique()

for fold, (train_idx, test_idx) in enumerate(tss.split(unique_hours)):
    train_hours = unique_hours[train_idx]
    test_hours = unique_hours[test_idx]

    train_df = df[df["time"].isin(train_hours)]
    test_df = df[df["time"].isin(test_hours)]

    # train your model here
    print(f"Fold {fold}: train {train_df['time'].min()} → {train_df['time'].max()}")
    print(f"         test  {test_df['time'].min()} → {test_df['time'].max()}")